In [1]:
%cd /datadrive/mount/MMMM

/datadrive/mount/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")
import time

In [6]:
# Import Pytorch
import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Import Pretrain Libraries (transformers + diffusers)
from transformers import BartTokenizer
from diffusers import AutoencoderKL

# Parallel Helper
# from parallel import DataParallelModel, DataParallelCriterion

# Import Our Own Functions
from master_init import *
from DSG import *

from count_params import count_params

In [7]:
torch.set_default_dtype(torch.float32)

# config = get_config() Implement later
print(f"[INFO] FETCHING CONFIGURATIONS.")
config = {
    "device" : "cuda",
    "device_ids" : [0,1,2,3],
    "staging_device" : "cuda",
    "num_epochs" : 50,
    "use_non_pytorch_parallel" : False,
    "test_run" : False,
    "live_evaluate" : True,
    "eval_interval" : 1,
    "log_dir" : "./logs"
}

device = config["device"]
device_ids = config["device_ids"]
staging_device = config["staging_device"]
num_epochs = config["num_epochs"]
use_non_pytorch_parallel = config["use_non_pytorch_parallel"]
test_run = config["test_run"]
live_evaluate = config["live_evaluate"]
eval_interval = config["eval_interval"]
log_dir = config["log_dir"]

print(f"[INFO] FINISHED CONFIGURATIONS.")

[INFO] FETCHING CONFIGURATIONS.
[INFO] FINISHED CONFIGURATIONS.


In [5]:
print(f"[INFO] INITIALIZING MODEL.")
model = INITIALIZE_MODEL(device=None, device_ids=device_ids, dtype=torch.float32)
if use_non_pytorch_parallel:
    model = DataParallelModel(model, device_ids=device_ids).to(device)
else:
    model = nn.DataParallel(model, device_ids=device_ids).to(device)

[INFO] INITIALIZING MODEL.
LOADED EEG ENCODER
LOADED CLIP ENCODER
LOADED BART MODEL
LOADED EEG-TEXT-BART
LOADED U-NET
LOADED CLIP TOKENIZER
LOADED EEG-IMG-DIFFUSION
LOADED EEG-IMG-CLASSIFICATION
LOADED EEG-TEXT-SENTIMENT


In [17]:
CLIP_text_encoder = 0
eeg_encoder = 0
emb_unet = 0
EEG_TEXT_BART_SENTIMENT_branch = 0
EEG_TEXT_BART_branch = 0
EEG_IMG_DIFFUSION_branch = 0
EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_branch = 0
meta_head = 0
EEG_TEXT_BART_head = 0
EEG_IMG_DIFFUSION_head = 0
EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_head = 0
for name, param in model.named_parameters():
    if "CLIP_text_encoder.text_model" in name:
        CLIP_text_encoder += param.numel()
        continue
    elif "eeg_encoder" in name:
        eeg_encoder += param.numel()
        continue
    elif "emb_unet" in name:
        emb_unet += param.numel()
        continue
    elif "branches.EEG-TEXT-BART-SENTIMENT" in name:
        EEG_TEXT_BART_SENTIMENT_branch += param.numel()
        continue
    elif "branches.EEG-TEXT-BART" in name:
        EEG_TEXT_BART_branch += param.numel()
        continue
    elif "branches.EEG-IMG-DIFFUSION" in name:
        EEG_IMG_DIFFUSION_branch += param.numel()
        continue
    elif "branches.EEG-IMG-BRAIN2IMAGE-CLASSIFICATION" in name:
        EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_branch += param.numel()
        continue
    elif "meta_head" in name:
        meta_head += param.numel()
#     elif "heads.EEG-TEXT-BART" in name:
#         EEG_TEXT_BART_head += param.numel()
#     elif "heads.EEG-IMG-DIFFUSION" in name:
#         EEG_IMG_DIFFUSION_head += param.numel()
#     elif "heads.EEG-IMG-BRAIN2IMAGE-CLASSIFICATION" in name:
#         EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_head += param.numel()
#     else:
    print(name, param.numel())

module.meta_head.hidden_layers.0.weight 786432
module.meta_head.hidden_layers.0.bias 1024
module.meta_head.hidden_layers.1.weight 1048576
module.meta_head.hidden_layers.1.bias 1024
module.meta_head.hidden_layers.2.weight 1048576
module.meta_head.hidden_layers.2.bias 1024
module.meta_head.hidden_layers.3.weight 1048576
module.meta_head.hidden_layers.3.bias 1024
module.meta_head.output_layer.weight 786432
module.meta_head.output_layer.bias 768
module.heads.EEG-TEXT-BART.hidden_layers.0.weight 786432
module.heads.EEG-TEXT-BART.hidden_layers.0.bias 1024
module.heads.EEG-TEXT-BART.hidden_layers.1.weight 1048576
module.heads.EEG-TEXT-BART.hidden_layers.1.bias 1024
module.heads.EEG-TEXT-BART.hidden_layers.2.weight 1048576
module.heads.EEG-TEXT-BART.hidden_layers.2.bias 1024
module.heads.EEG-TEXT-BART.hidden_layers.3.weight 1048576
module.heads.EEG-TEXT-BART.hidden_layers.3.bias 1024
module.heads.EEG-TEXT-BART.output_layer.weight 786432
module.heads.EEG-TEXT-BART.output_layer.bias 768
module.h

In [18]:
print(f"CLIP_text_encoder: {CLIP_text_encoder:,}")
print(f"eeg_encoder: {eeg_encoder:,}")
print(f"emb_unet: {emb_unet:,}")
print(f"EEG_TEXT_BART_SENTIMENT_branch: {EEG_TEXT_BART_SENTIMENT_branch:,}")
print(f"EEG_TEXT_BART_branch: {EEG_TEXT_BART_branch:,}")
print(f"EEG_IMG_DIFFUSION_branch: {EEG_IMG_DIFFUSION_branch:,}")
print(f"EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_branch: {EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_branch:,}")
print(f"meta_head: {meta_head:,}")
print(f"EEG_TEXT_BART_head: {EEG_TEXT_BART_head:,}")
print(f"EEG_IMG_DIFFUSION_head: {EEG_IMG_DIFFUSION_head:,}")
print(f"EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_head: {EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_head:,}")

CLIP_text_encoder: 123,060,480
eeg_encoder: 107,747,072
emb_unet: 3,406,721
EEG_TEXT_BART_SENTIMENT_branch: 4,989,443
EEG_TEXT_BART_branch: 409,308,160
EEG_IMG_DIFFUSION_branch: 861,095,620
EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_branch: 5,027,368
meta_head: 4,723,456
EEG_TEXT_BART_head: 0
EEG_IMG_DIFFUSION_head: 0
EEG_IMG_BRAIN2IMAGE_CLASSIFICATION_head: 0


In [23]:
eeg_encoder+meta_head+EEG_TEXT_BART_SENTIMENT_branch

117459971

In [18]:
eeg_encoder+CLIP_text_encoder

230807552

In [24]:
for name, param in model.named_parameters():
    param.requires_grad=True
    if ("eeg_encoder" in name) or ("emb_unet" in name) or ("CLIP_text_encoder" in name):
        param.requires_grad=False
        continue
    if "branches" in name:
        if "EEG-TEXT-BART.body" in name:
            if not (("lora" in name) or ("encoder.layers.0" in name) or ("embed_positions" in name) or ("shared" in name)):
                param.requires_grad=False
                continue
        if "EEG-IMG-DIFFUSION.body" in name:
            if not ("lora" in name):
                param.requires_grad=False
                continue
    if "CLIP_text_encoder" in name:
        param.requires_grad=False

In [25]:
def count_params(model, trainable=None):
    if trainable == None:
        return sum(p.numel() for p in model.parameters())
    elif trainable:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    elif not trainable:
        return sum(p.numel() for p in model.parameters() if not p.requires_grad)
        

In [26]:
count_params(model)

1538252144

In [27]:
count_params(model, True)

104394283

In [28]:
count_params(model, False)

1433857861

In [48]:
1539846512-123060480

1416786032